<a href="https://colab.research.google.com/github/omark243/Big-Data-/blob/main/Assinment%203.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Airbnb Price Prediction") \
    .getOrCreate()

In [7]:
df = spark.read.csv(
    "airbnb_price_prediction_sample (1).csv",
    header=True,
    inferSchema=True
)

In [8]:
df.printSchema()
df.show(5)
df.select("bathrooms", "bedrooms", "price").show()

root
 |-- id: integer (nullable = true)
 |-- bathrooms: double (nullable = true)
 |-- bedrooms: double (nullable = true)
 |-- beds: double (nullable = true)
 |-- accommodates: integer (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- number_of_reviews: integer (nullable = true)
 |-- review_scores_rating: double (nullable = true)
 |-- property_type: string (nullable = true)
 |-- room_type: string (nullable = true)
 |-- neighborhood: string (nullable = true)
 |-- price: double (nullable = true)

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|   room_type|   neighborhood| price|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+------------+---------------+------+
|  1|      1.0|     1.0| 2.0|           

In [9]:
from pyspark.sql.functions import col, sum

df.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).show()

+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------+------------+-----+
| id|bathrooms|bedrooms|beds|accommodates|minimum_nights|number_of_reviews|review_scores_rating|property_type|room_type|neighborhood|price|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------+------------+-----+
|  0|        0|       0|   0|           0|             0|                0|                   0|            0|        0|           0|    0|
+---+---------+--------+----+------------+--------------+-----------------+--------------------+-------------+---------+------------+-----+



In [10]:
dataDF = df.select("bathrooms", "bedrooms", "price")

In [11]:
dataDF = dataDF.dropna()
dataDF.show()

+---------+--------+------+
|bathrooms|bedrooms| price|
+---------+--------+------+
|      1.0|     1.0|146.65|
|      3.0|     1.0| 252.6|
|      3.0|     1.0|274.16|
|      1.0|     1.0|168.71|
|      4.0|     3.0|431.81|
|      1.0|     1.0|195.22|
|      2.5|     1.0|227.21|
|      1.5|     1.0|259.44|
|      1.0|     1.0|149.63|
|      4.0|     2.0|363.65|
|      3.0|     3.0|429.59|
|      1.0|     4.0|450.65|
|      1.0|     4.0|429.74|
|      4.0|     1.0|319.03|
|      2.0|     4.0|482.72|
|      2.5|     2.0|346.94|
|      1.0|     1.0|252.54|
|      1.0|     2.0|272.25|
|      3.0|     4.0|504.45|
|      1.0|     3.0|362.57|
+---------+--------+------+
only showing top 20 rows


In [12]:
trainDF, testDF = dataDF.randomSplit([0.8, 0.2], seed=42)
print("Training Rows:", trainDF.count())
print("Testing Rows:", testDF.count())

Training Rows: 98
Testing Rows: 22


In [13]:
from pyspark.ml.feature import VectorAssembler

vecAssembler = VectorAssembler(
    inputCols=["bathrooms", "bedrooms"],
    outputCol="features"
)

In [14]:
vecTrainDF = vecAssembler.transform(trainDF)
vecTrainDF.select("bathrooms", "bedrooms", "features", "price").show()

+---------+--------+---------+------+
|bathrooms|bedrooms| features| price|
+---------+--------+---------+------+
|      1.0|     1.0|[1.0,1.0]|144.23|
|      1.0|     1.0|[1.0,1.0]|146.65|
|      1.0|     1.0|[1.0,1.0]|157.92|
|      1.0|     1.0|[1.0,1.0]|158.41|
|      1.0|     1.0|[1.0,1.0]|164.55|
|      1.0|     1.0|[1.0,1.0]| 169.1|
|      1.0|     1.0|[1.0,1.0]|199.55|
|      1.0|     1.0|[1.0,1.0]|207.93|
|      1.0|     1.0|[1.0,1.0]|210.07|
|      1.0|     1.0|[1.0,1.0]|215.52|
|      1.0|     1.0|[1.0,1.0]| 232.9|
|      1.0|     1.0|[1.0,1.0]|233.02|
|      1.0|     1.0|[1.0,1.0]|252.54|
|      1.0|     2.0|[1.0,2.0]|255.01|
|      1.0|     2.0|[1.0,2.0]|267.16|
|      1.0|     2.0|[1.0,2.0]|272.12|
|      1.0|     2.0|[1.0,2.0]|272.25|
|      1.0|     2.0|[1.0,2.0]|274.42|
|      1.0|     2.0|[1.0,2.0]|293.59|
|      1.0|     2.0|[1.0,2.0]| 297.9|
+---------+--------+---------+------+
only showing top 20 rows


In [15]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction"
)

In [16]:
lrModel = lr.fit(vecTrainDF)

In [17]:
predDF = lrModel.transform(vecTrainDF)
predDF.select(
    "bathrooms",
    "bedrooms",
    "features",
    "price",
    "prediction"
).show()

+---------+--------+---------+------+-----------------+
|bathrooms|bedrooms| features| price|       prediction|
+---------+--------+---------+------+-----------------+
|      1.0|     1.0|[1.0,1.0]|144.23|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|146.65|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|157.92|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|158.41|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|164.55|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]| 169.1|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|199.55|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|207.93|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|210.07|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|215.52|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]| 232.9|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|233.02|198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|252.54|198.6362083457284|
|      1.0|     2.0|[1.0,2.0]|255.01|280.7459323705309|
|      1.0|     2.0|[1.0,2.0]|267.16|280.7459323

In [18]:
vecTestDF = vecAssembler.transform(testDF)
predTestDF = lrModel.transform(vecTestDF)
predTestDF.select(
    "bathrooms",
    "bedrooms",
    "features",
    "price",
    "prediction"
).show()

+---------+--------+---------+------+------------------+
|bathrooms|bedrooms| features| price|        prediction|
+---------+--------+---------+------+------------------+
|      1.0|     1.0|[1.0,1.0]|149.63| 198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|168.71| 198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|195.22| 198.6362083457284|
|      1.0|     1.0|[1.0,1.0]|229.58| 198.6362083457284|
|      1.0|     2.0|[1.0,2.0]|269.29| 280.7459323705309|
|      1.0|     2.0|[1.0,2.0]| 287.6| 280.7459323705309|
|      1.0|     2.0|[1.0,2.0]|338.83| 280.7459323705309|
|      1.0|     3.0|[1.0,3.0]|372.45|362.85565639533337|
|      1.0|     4.0|[1.0,4.0]|484.57| 444.9653804201359|
|      1.5|     1.0|[1.5,1.0]|218.04|215.18715038797325|
|      1.5|     1.0|[1.5,1.0]|220.54|215.18715038797325|
|      1.5|     1.0|[1.5,1.0]|259.44|215.18715038797325|
|      1.5|     2.0|[1.5,2.0]|270.38| 297.2968744127757|
|      1.5|     2.0|[1.5,2.0]| 300.2| 297.2968744127757|
|      2.0|     1.0|[2.0,1.0]|2

In [19]:
from pyspark.ml.evaluation import RegressionEvaluator

rmseEvaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="rmse"
)

r2Evaluator = RegressionEvaluator(
    labelCol="price",
    predictionCol="prediction",
    metricName="r2"
)

rmse = rmseEvaluator.evaluate(predTestDF)
r2 = r2Evaluator.evaluate(predTestDF)

print("RMSE:", rmse)
print("R2:", r2)

RMSE: 34.2336517144055
R2: 0.9027306123012013


In [20]:
print("Intercept:", lrModel.intercept)
print("Coefficients:", lrModel.coefficients)

Intercept: 83.42460023643623
Coefficients: [33.10188408448968,82.10972402480249]


In [21]:
improvedDF = df.select(
    "bathrooms",
    "bedrooms",
    "beds",
    "accommodates",
    "review_scores_rating",
    "price"
).dropna()

In [22]:
trainImprovedDF, testImprovedDF = improvedDF.randomSplit([0.8, 0.2], seed=42)

In [23]:
improvedAssembler = VectorAssembler(
    inputCols=[
        "bathrooms",
        "bedrooms",
        "beds",
        "accommodates",
        "review_scores_rating"
    ],
    outputCol="features"
)

In [24]:
vecTrainImprovedDF = improvedAssembler.transform(trainImprovedDF)

improvedLR = LinearRegression(
    featuresCol="features",
    labelCol="price",
    predictionCol="prediction"
)

improvedModel = improvedLR.fit(vecTrainImprovedDF)

In [25]:
vecTestImprovedDF = improvedAssembler.transform(testImprovedDF)
predImprovedDF = improvedModel.transform(vecTestImprovedDF)
predImprovedDF.select(
    "bathrooms",
    "bedrooms",
    "beds",
    "accommodates",
    "review_scores_rating",
    "price",
    "prediction"
).show()

+---------+--------+----+------------+--------------------+------+------------------+
|bathrooms|bedrooms|beds|accommodates|review_scores_rating| price|        prediction|
+---------+--------+----+------------+--------------------+------+------------------+
|      1.0|     1.0| 1.0|           2|                74.7|164.55|171.84049020576933|
|      1.0|     1.0| 1.0|           4|                72.2|157.92| 192.2730066357054|
|      1.0|     1.0| 1.0|           4|                91.7|229.58| 223.1201779680172|
|      1.0|     1.0| 2.0|           3|                74.2|146.65|187.60104572985742|
|      1.0|     2.0| 2.0|           6|                80.3|255.01| 285.7015323127442|
|      1.0|     2.0| 3.0|           4|                90.2|274.42| 281.3329895693576|
|      1.0|     2.0| 3.0|           6|                98.2|338.83|318.37552133207697|
|      1.0|     3.0| 4.0|           7|                94.0|371.83|380.15295948374643|
|      1.0|     4.0| 5.0|          10|                

In [26]:
improvedRMSE = rmseEvaluator.evaluate(predImprovedDF)
improvedR2 = r2Evaluator.evaluate(predImprovedDF)

print("Improved RMSE:", improvedRMSE)
print("Improved R2:", improvedR2)

Improved RMSE: 22.776053169327042
Improved R2: 0.9570699448939194


In [27]:
print("The first model used only bathrooms and bedrooms to predict the price.")
print("The improved model used more features such as beds, accommodates, and review_scores_rating.")
print("Using more relevant features can help improve the prediction accuracy.")

The first model used only bathrooms and bedrooms to predict the price.
The improved model used more features such as beds, accommodates, and review_scores_rating.
Using more relevant features can help improve the prediction accuracy.
